# Mix Repair (v2): token-exact blends of `think-midtrain` (Stage 5) + `think-dataset-clean-1930s`

Rebuilds the `mixed/ratio_<pct>` mixtures under **`mixed/v2/ratio_<pct>/`**, fixing the flaw in
`midtraining_mix_ratios.ipynb`: v1 hit its ratio by **document count**, and because midtrain
documents are ~2.7x shorter than old-corpus books (and the midtrain pool ran out), `ratio_30`
landed at ~11.5% by docs and ~4.5% by characters. This notebook targets the ratio by **tokens**
under the run's exact tokenizer, and never touches the v1 folders.

What changed vs `midtraining_mix_ratios.ipynb`:

| | v1 (`mixed/ratio_<pct>`) | v2 (`mixed/v2/ratio_<pct>`) |
|---|---|---|
| Sizing | `TARGET_TOTAL_DOCS` | `EXPECTED_CONSUMED_TOKENS` x (1 + cache margin) |
| Ratio basis | documents | **tokens** (exact tokenizer, BOS counted per doc) |
| Old-corpus start | shard 0, seeded random shard order | `OLD_START_PCT` through the corpus, **forward only** |
| Midtrain reuse | sample without replacement | cyclic from `MIDTRAIN_START_PCT`, capped by `MIDTRAIN_MAX_EPOCHS` |
| Ratio over prefixes | approximate (global shuffle) | ratio-tracking interleave: every prefix is on target |
| Validation | last shard of the mix | dedicated held-out shard, disjoint from train |
| Bookkeeping | doc/char manifest | per-shard sidecars: exact tokens, sha256, doc ids, text hashes |

Flow: load the exact tokenizer -> index both sources (char length + normalized-text hash, exact
tokens on a systematic sample to calibrate chars/token) -> dedupe -> reserve a validation shard ->
interleave the two streams so the **running token ratio tracks the target at every prefix** ->
plan shards by estimated tokens -> pack by streaming source shards through an LRU, exact-tokenizing
every document as it is written -> verify achieved ratios exactly -> manifest + README.

Sizing is estimate-then-verify: planning uses a chars/token calibration measured on a systematic
sample of the real documents, and every written document is then exact-tokenized so the manifest
reports **measured** token counts and ratios, not estimates. If the achieved ratio misses
`RATIO_TOLERANCE_PP`, section 9 prints refined chars/token constants to pin in `CPT_OVERRIDE`.

`RANDOM_SEED` fixes every shuffle; reruns skip shards already uploaded.

In [ ]:
%pip -q install -U huggingface_hub pyarrow tqdm numpy tiktoken

import os, json, re, time, random, pickle, hashlib, tempfile, shutil
from collections import defaultdict, OrderedDict

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import tqdm
from huggingface_hub import (
    HfApi, hf_hub_download, list_repo_files,
    CommitOperationAdd, CommitOperationDelete,
)
from huggingface_hub.utils import HfHubHTTPError
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN_Z')
api = HfApi(token=HF_TOKEN)

MIDTRAIN_REPO = 'zachnorton03/think-midtrain'    # destination repo AND midtrain source
OLD_REPO = 'jbduran/think-dataset-clean-1930s'   # old/original source (public, read-only)
MIDTRAIN_SRC_PREFIX = 'stage5/data'              # midtrain source shards

# The run's tokenizer lives next to its checkpoints. It is loaded (never retrained) and used to
# measure every token count in this notebook, so composition is reported in the same units the
# trainer consumes.
TOKENIZER_REPO = 'jbduran/think.nano'
TOKENIZER_TAG = 'clean1930s-d24-r12-ctx4096-sssl-fulltok-v1'
TOKENIZER_PREFIX = f'experiments/{TOKENIZER_TAG}/tokenizer'

os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '30')
DL_MAX_RETRIES = 5
DL_ETAG_TIMEOUT = 30


def list_repo_paths(repo_id, token=None, repo_type='dataset'):
    return set(list_repo_files(repo_id=repo_id, repo_type=repo_type, token=token))


def robust_hf_download(path_in_repo, repo_id, token=None, repo_type='dataset'):
    """hf_hub_download with a bounded per-attempt timeout + retry, so a hung CDN node raises
    instead of blocking forever."""
    last_exc = None
    for attempt in range(1, DL_MAX_RETRIES + 1):
        try:
            return hf_hub_download(
                repo_id=repo_id, repo_type=repo_type, filename=path_in_repo,
                token=token, etag_timeout=DL_ETAG_TIMEOUT, force_download=(attempt > 1),
            )
        except Exception as exc:
            last_exc = exc
            print(f"  download retry {attempt}/{DL_MAX_RETRIES} for {repo_id}/{path_in_repo}: {exc}")
    raise last_exc


def robust_commit(operations, commit_message, max_retries=6):
    """api.create_commit with backoff that honors HF's 'Retry after N seconds' on 429s.

    One create_commit == one repo commit, no matter how many files it carries -- this is what
    keeps us under the 128-commits/hour limit. Returns the CommitInfo, whose `oid` is the
    immutable commit SHA we report at the end of the run."""
    for attempt in range(1, max_retries + 1):
        try:
            return api.create_commit(
                repo_id=MIDTRAIN_REPO, repo_type='dataset',
                operations=operations, commit_message=commit_message,
            )
        except HfHubHTTPError as exc:
            status = getattr(getattr(exc, 'response', None), 'status_code', None)
            if status != 429 or attempt == max_retries:
                raise
            m = re.search(r'[Rr]etry (?:this action |after )?in ?(\d+)\s*(second|minute)', str(exc))
            wait = int(m.group(1)) * (60 if m and m.group(2) == 'minute' else 1) if m else 60 * attempt
            wait += 5  # small cushion past the window boundary
            tqdm.tqdm.write(f"  429 rate-limited; sleeping {fmt_secs(wait)} before retry "
                            f"{attempt + 1}/{max_retries}...")
            time.sleep(wait)
    raise RuntimeError("robust_commit exhausted retries")  # unreachable


def read_text_column(local_parquet):
    return pq.read_table(local_parquet, columns=['text']).column('text').to_pylist()


def shard_index_of(path):
    m = re.search(r'shard_(\d+)\.parquet$', path)
    assert m, f"Not a shard filename: {path}"
    return int(m.group(1))


def norm_hash(text):
    """Stable content hash for dedup: lowercased, whitespace-collapsed text. Catches the same
    book re-OCR'd with different line wrapping, which a raw byte hash would miss."""
    return hashlib.sha1(' '.join(text.lower().split()).encode('utf-8', 'ignore')).hexdigest()


def doc_id(d):
    """Stable document id: which source shard, which row."""
    return f"{d['source']}:{d['shard_path']}#{d['row_idx']}"


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()


def fmt_secs(s):
    m, s = divmod(int(s), 60)
    h, m = divmod(m, 60)
    return f"{h}h{m:02d}m" if h else f"{m}m{s:02d}s"

## 1. Configuration -- edit these for each mixture, then run top to bottom

One mixture per run. The two mixtures for the d32 run are pre-written below; uncomment the one you
are building (or write your own).

The old corpus is consumed **forward only** from `OLD_START_PCT` through the shard list, so the two
mixtures draw from disjoint regions (r30 from 70%, r60 from 90%) and neither touches the base run's
pretraining shards. Shard 472 is excluded by default: it is the base run's *validation* shard
(`experiment_tokenizer.json` records `validation_shard: 472`).

The midtrain corpus is small enough to repeat. `MIDTRAIN_START_PCT` picks where in the (shuffled)
midtrain document order to start and wraps around the end; `MIDTRAIN_MAX_EPOCHS` caps how much of
the corpus that consumption may cover in total, and the run **aborts before downloading anything
big** if the plan exceeds it. Starting at 0.50 with a 0.40 cap is fine -- it consumes 50% -> 90%.

**Measured inputs (no need to re-derive).** The midtrain corpus is **608,471,853 tokens** across
11,409 documents (2,543,809,124 chars, 4.1807 chars/token) under this run's exact tokenizer, with
no duplicates, no empty documents and no text overlap with the old corpus. The old corpus averages
~51.4M tokens per shard over 473 shards. Stage 2 consumes 4,026,531,840 tokens and stage 3
consumes 2,013,265,920, per the run's mixture schedule.

**Selected plan: 21% for stage 2, 45% for stage 3** -- 1,751,541,350 midtrain tokens total, which
is **8.7000% of the run** and **2.990 epochs** over the pool (2.879 as the model actually sees it),
against a 3.00 cap. This is preset A's gradual ramp with stage 2 lifted from 0.20 to 0.21 so the
total matches the 8.70% budget; stage 3 keeps its 0.45. Both stages draw from the same pool, so
their epoch counts add -- `MIDTRAIN_MAX_EPOCHS` in the cell below is the per-mixture guard, not
the total.


In [ ]:
# =============================== WHAT TO BUILD ================================================
# One mixture per run: build stage 2 first, upload it, then swap in the stage 3 block below.
#
# ---------------------------------------------------------------------------------------------
# MIDTRAIN CORPUS -- zachnorton03/think-midtrain, stage5/data @ 9ace24b8, measured exactly with
# this run's tokenizer (BOS counted per document, as pretok_think prepends it):
#
#       608,471,853 tokens | 11,409 docs | 2,543,809,124 chars | 4.1807 chars/token
#
# No duplicate documents, no empty documents, and zero normalized-text overlap with the old
# corpus, so section 5's dedup removes nothing: the whole 608,471,853 is usable pool and the
# epoch figures below are exact rather than upper bounds.
# ---------------------------------------------------------------------------------------------
#
# How much each stage consumes comes from the run's own mixture schedule
# (experiments/Think.Unbounded-d32/config.json), which is the source of truth:
#
#       total_tokens                                                       20,132,659,200
#       stage 2  "injection"   14,092,861,440 -> 18,119,393,280          =  4,026,531,840
#       stage 3  "decay_mix"   18,119,393,280 -> 20,132,659,200          =  2,013,265,920
#
# SELECTED PLAN -- preset A "gradual ramp", topped up to 8.70% total midtrain usage:
#
#                ratio     consumed        midtrain seen    epochs built   epochs seen
#     stage 2     21%     4,026,531,840      845,571,686       1.439          1.390
#     stage 3     45%     2,013,265,920      905,969,664       1.551          1.489
#     total                                1,751,541,350       2.990          2.879
#
# Midtrain is 8.7000% of the 20.13B-token run -- the same total budget presets B and C spend,
# reached here by lifting stage 2 from 0.20 to 0.21 and leaving stage 3's 0.45 untouched, so the
# gradual ramp is preserved. Ratios are BY TOKENS, not by documents or characters: at these
# settings stage 2 is ~20.6% by characters and ~43.7% by document count, because old-corpus books
# average 155,444 tokens against midtrain's 53,333.
#
# The two mixtures are built independently and both draw from the whole stage 5 pool, so the
# run's real reuse is the SUM of the two stages -- that sum is what the 3-epoch cap applies to.
# "built" epochs include the 3% cache margin and the held-out validation shard, and are what the
# MIDTRAIN_MAX_EPOCHS guard measures; "seen" epochs are what the model actually trains on.
#
# Alternatives at the same cap -- swap BOTH stages together, never mix presets:
#     B balanced      stage2 0.15 / stage3 0.57  ->  1.028 + 1.964 = 2.992  (1.752B seen)
#     C late-heavy    stage2 0.10 / stage3 0.67  ->  0.685 + 2.309 = 2.994  (1.752B seen)
# The r30/r60 pair this notebook originally shipped is 4.124 epochs (2.056 + 2.068), over the
# cap and consistent with mixture_schedule.max_epochs = 4 in the parent run's config.
#
# ---------------------------------------------------------------------------------------------
# CHARS/TOKEN CALIBRATION (CPT_OVERRIDE) -- per stage, never shared between them.
# The planner sizes each document as chars / chars_per_token, calibrated on every 8th document
# while indexing. Section 9 then exact-tokenizes everything written and fails if the achieved
# ratio misses RATIO_TOLERANCE_PP. Stage 3's first pass missed by +0.1486 pp: midtrain's real
# tokens came in 0.73% above the estimate while the old corpus came in only 0.32% above, and that
# gap tilts the ratio toward midtrain. Pinning the measured constants makes the rebuild plan from
# truth instead of from the sample.
#     stage 3 (r45): {'midtrain': 4.1571, 'old_corpus': 4.3154}   measured, first pass
#     stage 2 (r21): {}                                           not calibrated yet
# These are NOT interchangeable. Each stage draws a different midtrain slice (its own
# MIDTRAIN_START_PCT and epoch count) and a different set of old-corpus shards, so each has its
# own chars/token. Carrying stage 3's constants into stage 2 would bias stage 2's ratio.
#
# ---------------------------------------------------------------------------------------------
# VARIABLES TO SET, PER STAGE                     stage 2          stage 3
#   RATIO_MIDTRAIN                                0.21             0.45
#   EXPECTED_CONSUMED_TOKENS                      4_026_531_840    2_013_265_920
#   OLD_START_PCT                                 0.70             0.90
#   MIDTRAIN_START_PCT                            0.00             0.44
#   MIDTRAIN_MAX_EPOCHS                           1.60             1.75
#   CPT_OVERRIDE                                  {}               see calibration note above
#   VAL_TOKENS                                    20_971_520       (same)
#   CACHE_MARGIN_FRAC                             0.03             (same)
#   VAL_MARGIN_FRAC                               0.10             (same)
#   TOKENS_PER_SHARD                              64_000_000       (same)
#   RANDOM_SEED                                   42               (same -- MUST match)
#   GENERATOR_CODE_COMMIT                         set before building, both stages
#
# RANDOM_SEED must be identical across the two builds: it is what makes mid_order the SAME
# document sequence in each, which is what gives MIDTRAIN_START_PCT its meaning. Stage 3's
# 0.44 is the fractional part of stage 2's built epoch count (1.439), so stage 3 picks up where
# stage 2 stopped instead of re-showing the documents stage 2 opened with.
#
# MIDTRAIN_MAX_EPOCHS is a PER-MIXTURE guard (this notebook only ever sees one stage at a time),
# set ~12% above the planned value so it still catches a misconfiguration without tripping on
# the docs-vs-tokens skew in the epochs_docs half of the check. It is not the 2.990 total.
#
# OLD CORPUS -- 473 shards, ~51.4M tokens each; the base run used 0..329 and 472 is its
# validation shard, so both stages draw forward out of what is left:
#     stage 2 from shard 331 (94 available), needs ~64 at ratio 0.21 (~67 indexed with slack,
#             reaching shard 397, clear of stage 3's region)
#     stage 3 from shard 425 (47 available), needs ~22 at ratio 0.45
# A LOWER midtrain ratio means MORE old corpus, so recheck this if you change the ratios.
#
# Downstream, the run's config.json needs datasets.<source>.subfolder pointed at
# mixed/v2/ratio_21/data and mixed/v2/ratio_45/data, with revision set to the immutable commit
# SHA printed at the end of each build and num_train_shards / validation_shard set to the counts
# printed by section 6. See d32-branch-sample-config.json in this repo.
#
# Build order does not matter: the stagger is about which slice of the shuffled midtrain order
# each stage covers (stage 2 spans 0.00->1.439, stage 3 spans 0.44->2.011), not about which runs
# first. Stage 3 was built first here.
#
# ---- ACTIVE: stage 3, "decay_mix" -> mixed/v2/ratio_45 --------------------------------------
RATIO_MIDTRAIN = 0.45              # midtrain share of the mixture, BY TOKENS (0.0 - 1.0)
EXPECTED_CONSUMED_TOKENS = 2_013_265_920   # train tokens the run will actually consume from this mix
OLD_START_PCT = 0.90               # start this far through the old corpus's shard list (forward only)
MIDTRAIN_START_PCT = 0.44          # picks up where stage 2's 1.439 epochs leave off
MIDTRAIN_MAX_EPOCHS = 1.75         # abort if THIS mixture would consume more midtrain than this
CPT_OVERRIDE = {'midtrain': 4.1571, 'old_corpus': 4.3154}   # measured on this stage's own draw
#
# ---- stage 2, "injection" -> mixed/v2/ratio_21: swap in for the six lines above --------------
# RATIO_MIDTRAIN = 0.21
# EXPECTED_CONSUMED_TOKENS = 4_026_531_840
# OLD_START_PCT = 0.70
# MIDTRAIN_START_PCT = 0.00
# MIDTRAIN_MAX_EPOCHS = 1.60
# CPT_OVERRIDE = {}                # stage 2 is uncalibrated -- start empty, pin what section 9 prints

# Build enough that the trainer's cache never runs dry on the tail of the run.
CACHE_MARGIN_FRAC = 0.03           # uploaded train tokens = EXPECTED_CONSUMED_TOKENS * (1 + this)
VAL_TOKENS = 20_971_520            # = pretokenize.val_tokens, NOT training.eval_tokens (2_097_152).
                                   # experiment.py passes pretokenize.val_tokens to pretok_think as
                                   # --val-tokens; _existing_cache_satisfies rejects any cache whose
                                   # val_tokens is under it, so a short validation shard silently
                                   # re-pretokenizes the whole stage on every resume.
VAL_MARGIN_FRAC = 0.10             # build the val shard this much over the floor

RANDOM_SEED = 42                   # fixes every shuffle + the interleave; recorded in the manifest
TOKENS_PER_SHARD = 64_000_000      # ~250M chars/shard, the convention Stages 2-5 use

# Ratio accuracy. Planning uses a chars/token calibration; section 9 measures what was actually
# written. If |achieved - target| exceeds this, section 9 fails and prints the exact chars/token
# constants to pin in CPT_OVERRIDE for a clean rebuild. CPT_OVERRIDE is set PER STAGE, up in the
# ACTIVE block -- it describes one stage's specific draw and must not be carried across stages.
RATIO_TOLERANCE_PP = 0.10          # percentage points
CALIB_SAMPLE_EVERY = 8             # exact-tokenize every Nth doc while indexing (1 = every doc)

# --- Sources / layout ------------------------------------------------------------------------
OLD_EXCLUDE_SHARDS = {472}         # base run's validation shard -- never mix it into training
BASE_RUN_OLD_SHARDS_UPTO = 329     # base run pretrained on old shards 0..this; must not be reused
OLD_INDEX_SLACK = 1.05             # index this much more old corpus than the plan needs
ENFORCE_DISJOINT_OLD_RANGE = True  # abort if another mixed/v2 mixture already used these old shards
SHUFFLE_WINDOW_OLD_SHARDS = 3      # docs are shuffled within a sliding window of this many shards
SHUFFLE_WINDOW_MIDTRAIN_SHARDS = 3

# --- Machine limits --------------------------------------------------------------------------
SOURCE_CACHE_SHARDS = 10           # parsed source shards held in RAM while packing (~200MB each)
TOKENIZE_BATCH_DOCS = 8            # docs per encode call; keeps token-id lists from ballooning RAM
TOKENIZER_THREADS = 8
SHARDS_PER_COMMIT = 20             # output shards per HF commit (HF caps dataset commits at 128/hr)

# --- Provenance ------------------------------------------------------------------------------
# Commit SHA of the repo holding THIS notebook, recorded in the manifest as the generator version.
GENERATOR_CODE_COMMIT = None       # e.g. '8d53d2c...' -- set it; None only warns
GENERATOR_CODE_REPO = 'think.nano-midtraining'
GENERATOR_NOTEBOOK = 'midtraining_mix_repair.ipynb'

# --- Derived (no need to edit) ---------------------------------------------------------------
assert 0.0 <= RATIO_MIDTRAIN <= 1.0, "RATIO_MIDTRAIN must be in [0, 1]."
assert 0.0 <= OLD_START_PCT < 1.0, "OLD_START_PCT must be in [0, 1)."
assert 0.0 <= MIDTRAIN_START_PCT < 1.0, "MIDTRAIN_START_PCT must be in [0, 1)."
assert MIDTRAIN_MAX_EPOCHS > 0, "MIDTRAIN_MAX_EPOCHS must be positive."

RATIO_PCT = round(RATIO_MIDTRAIN * 100)
DST_PREFIX = f'mixed/v2/ratio_{RATIO_PCT}'
DATA_PREFIX = f'{DST_PREFIX}/data'
SHARDMETA_PREFIX = f'{DST_PREFIX}/shard_meta'   # per-shard sidecars, kept OUT of data/

BUILD_TARGET_TOKENS = int(EXPECTED_CONSUMED_TOKENS * (1.0 + CACHE_MARGIN_FRAC))
VAL_TARGET_TOKENS = int(VAL_TOKENS * (1.0 + VAL_MARGIN_FRAC))
MIDTRAIN_TOKENS_NEEDED = BUILD_TARGET_TOKENS * RATIO_MIDTRAIN + VAL_TARGET_TOKENS * RATIO_MIDTRAIN
OLD_TOKENS_NEEDED = BUILD_TARGET_TOKENS * (1 - RATIO_MIDTRAIN) + VAL_TARGET_TOKENS * (1 - RATIO_MIDTRAIN)

print(f"Building {MIDTRAIN_REPO}/{DST_PREFIX}/")
print(f"  ratio:      {RATIO_PCT}% midtrain / {100 - RATIO_PCT}% old-corpus, BY TOKENS")
print(f"  consumed:   {EXPECTED_CONSUMED_TOKENS:,} train tokens "
      f"({EXPECTED_CONSUMED_TOKENS * RATIO_MIDTRAIN:,.0f} midtrain / "
      f"{EXPECTED_CONSUMED_TOKENS * (1 - RATIO_MIDTRAIN):,.0f} pretraining)")
print(f"  building:   {BUILD_TARGET_TOKENS:,} train tokens (+{CACHE_MARGIN_FRAC:.0%} cache margin) "
      f"+ a {VAL_TARGET_TOKENS:,}-token validation shard")
print(f"  sources:    old corpus from {OLD_START_PCT:.0%} forward (excluding shards "
      f"{sorted(OLD_EXCLUDE_SHARDS)}), midtrain from {MIDTRAIN_START_PCT:.0%} "
      f"(<= {MIDTRAIN_MAX_EPOCHS:.2f} epochs)")
print(f"  shards:     ~{BUILD_TARGET_TOKENS / TOKENS_PER_SHARD:.0f} train "
      f"@ {TOKENS_PER_SHARD:,} tokens + 1 validation | seed {RANDOM_SEED}")
if GENERATOR_CODE_COMMIT is None:
    print("  WARNING: GENERATOR_CODE_COMMIT is None -- the manifest will not record which "
          "revision of the notebook generated this dataset.")

## 1b. (Optional) Wipe everything under this mixture's v2 folder

Deletes all files under `mixed/v2/ratio_<pct>/` in a single commit so the mixture can be rebuilt
from scratch (needed if you change `RANDOM_SEED`, the ratio, the token targets, or the start
percentages after a partial upload). Guarded by `CONFIRM_WIPE`; leave it `None` to preview.

**This can only ever touch `mixed/v2/...`.** The v1 folders (`mixed/ratio_30`, `mixed/ratio_60`,
`mixed/ratio_0`) are asserted out of range -- they must be preserved.

In [ ]:
# Set CONFIRM_WIPE to the exact folder name (e.g. f'v2/ratio_{RATIO_PCT}') to delete; None = preview.
CONFIRM_WIPE = None

wipe_prefix = f'{DST_PREFIX}/'
assert wipe_prefix.startswith('mixed/v2/'), (
    f"Refusing to wipe {wipe_prefix}: this notebook only ever writes under mixed/v2/. The v1 "
    f"mixtures must be preserved.")

to_delete = sorted(f for f in list_repo_paths(MIDTRAIN_REPO, token=HF_TOKEN) if f.startswith(wipe_prefix))
print(f"{len(to_delete)} file(s) under {MIDTRAIN_REPO}/{wipe_prefix}")
for f in to_delete[:10]:
    print(f"  {f}")
if len(to_delete) > 10:
    print(f"  ... and {len(to_delete) - 10} more")

if not to_delete:
    print("Nothing to wipe.")
elif CONFIRM_WIPE != f'v2/ratio_{RATIO_PCT}':
    print(f"\nPREVIEW ONLY -- set CONFIRM_WIPE = 'v2/ratio_{RATIO_PCT}' and rerun to delete these.")
else:
    # One create_commit deletes all of them together = a single commit against the hourly cap.
    info = robust_commit(
        [CommitOperationDelete(path_in_repo=f) for f in to_delete],
        commit_message=f'Wipe v2/ratio_{RATIO_PCT} ({len(to_delete)} files)',
    )
    print(f"\nDeleted {len(to_delete)} file(s) from {MIDTRAIN_REPO}/{wipe_prefix} (commit {info.oid})")

## 2. Load the run's exact tokenizer

Downloads `experiments/<tag>/tokenizer/` from the checkpoint repo and unpickles `tokenizer.pkl`
into the `tiktoken` encoding the run was trained with (`rustbpe`, vocab 32,768) -- the same object
`nanochat.tokenizer.RustBPETokenizer.from_directory` loads, so counts here are the counts
`scripts/pretok_think.py` will produce.

**Every document counts its BOS.** `pretok_think` encodes with `prepend=<|bos|>`, so a document's
token cost is `1 + len(encode_ordinary(text))`. That +1 is included everywhere in this notebook.

The directory is fingerprinted with the same sha256-over-sorted-files recipe `pretok_think` uses,
and the fingerprint goes in the manifest so the measurement is reproducible.

In [ ]:
import tiktoken

TOKENIZER_DIR = os.path.join(tempfile.mkdtemp(), 'tokenizer')
os.makedirs(TOKENIZER_DIR, exist_ok=True)

tok_files = sorted(p for p in list_repo_paths(TOKENIZER_REPO, token=HF_TOKEN, repo_type='model')
                   if p.startswith(f'{TOKENIZER_PREFIX}/'))
assert tok_files, f"No tokenizer files under {TOKENIZER_REPO}/{TOKENIZER_PREFIX}/"
for repo_path in tok_files:
    cached = robust_hf_download(repo_path, repo_id=TOKENIZER_REPO, token=HF_TOKEN, repo_type='model')
    shutil.copy2(cached, os.path.join(TOKENIZER_DIR, os.path.basename(repo_path)))


def tokenizer_fingerprint(tokenizer_dir):
    """sha256 over (relative path, bytes) of every file, sorted -- identical recipe to
    scripts/pretok_think.py, so the fingerprint here matches the one recorded at pretokenization."""
    digest = hashlib.sha256()
    files = []
    for root, _, names in os.walk(tokenizer_dir):
        for name in names:
            path = os.path.join(root, name)
            files.append((os.path.relpath(path, tokenizer_dir), path))
    for relative, path in sorted(files):
        digest.update(relative.encode('utf-8'))
        digest.update(b'\0')
        with open(path, 'rb') as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b''):
                digest.update(chunk)
    return digest.hexdigest()


TOKENIZER_FINGERPRINT = tokenizer_fingerprint(TOKENIZER_DIR)

with open(os.path.join(TOKENIZER_DIR, 'tokenizer.pkl'), 'rb') as f:
    enc = pickle.load(f)
assert isinstance(enc, tiktoken.core.Encoding), f"tokenizer.pkl is a {type(enc)}, expected an Encoding"
BOS_ID = enc.encode('<|bos|>', allowed_special={'<|bos|>'})[0]
VOCAB_SIZE = enc.n_vocab


def token_counts(texts, num_threads=None):
    """Exact token count per document, including the BOS prepended to every document.

    Encodes in small batches and keeps only the lengths: a 64M-token shard's ids would be a
    multi-GB list of Python ints if materialized all at once."""
    threads = TOKENIZER_THREADS if num_threads is None else num_threads
    out = []
    for i in range(0, len(texts), TOKENIZE_BATCH_DOCS):
        ids = enc.encode_ordinary_batch(texts[i:i + TOKENIZE_BATCH_DOCS], num_threads=threads)
        out.extend(len(x) + 1 for x in ids)   # +1 = BOS
        del ids
    return out


_probe = 'Radio-frequency amplification in the 1937 laboratory notebook.'
_probe_tokens = token_counts([_probe])[0]
assert _probe_tokens == len(enc.encode_ordinary(_probe)) + 1
print(f"Tokenizer: {TOKENIZER_REPO}/{TOKENIZER_PREFIX} ({len(tok_files)} files, name={enc.name!r})")
print(f"  vocab_size={VOCAB_SIZE:,} | bos_id={BOS_ID} | fingerprint={TOKENIZER_FINGERPRINT[:16]}...")
print(f"  probe: {_probe_tokens} tokens for a {len(_probe)}-char string (BOS included)")

## 3. Index the midtrain corpus + calibrate chars/token

Reads every Stage 5 shard once, recording per document: source shard, row, character length, and a
normalized-text hash (for dedup in section 5). Every `CALIB_SAMPLE_EVERY`-th document is
exact-tokenized as it goes by -- the shard is already downloaded and parsed, so the calibration
sample is nearly free and large enough (~12% of the corpus at the default) that the chars/token
constant is tight.

The **midtrain epoch check runs here**, before the old corpus (tens of GB) is touched: if the
mixture would consume more than `MIDTRAIN_MAX_EPOCHS` of the midtrain corpus, this cell raises.

In [ ]:
CALIBRATION = {'midtrain': {'chars': 0, 'body_tokens': 0, 'docs': 0},
               'old_corpus': {'chars': 0, 'body_tokens': 0, 'docs': 0}}


def index_shard(local_parquet, shard_path, source_tag, calib_offset):
    """Index one source shard: char length + normalized hash per doc, exact tokens on a
    systematic sample. Returns (docs, n_sampled). Text is released before returning."""
    texts = read_text_column(local_parquet)
    docs, sample_rows = [], []
    for row_idx, raw in enumerate(texts):
        text = raw if isinstance(raw, str) else ''
        docs.append({'source': source_tag, 'shard_path': shard_path, 'row_idx': row_idx,
                     'char_len': len(text), 'hash': norm_hash(text)})
        if text and (calib_offset + row_idx) % CALIB_SAMPLE_EVERY == 0:
            sample_rows.append(row_idx)
    if sample_rows:
        sample = [texts[i] for i in sample_rows]
        counts = token_counts(sample)
        CALIBRATION[source_tag]['chars'] += sum(len(t) for t in sample)
        CALIBRATION[source_tag]['body_tokens'] += sum(counts) - len(counts)   # strip the BOS
        CALIBRATION[source_tag]['docs'] += len(counts)
        del sample, counts
    del texts
    return docs, len(sample_rows)


def chars_per_token(source_tag):
    """Chars per non-BOS token for a source, from the calibration sample (or CPT_OVERRIDE)."""
    if source_tag in CPT_OVERRIDE:
        return float(CPT_OVERRIDE[source_tag])
    c = CALIBRATION[source_tag]
    assert c['body_tokens'] > 0, f"No calibration sample yet for {source_tag}."
    return c['chars'] / c['body_tokens']


def est_tokens(doc, cpt_by_source):
    """Estimated cost of a document in the trainer's token stream: body + 1 BOS."""
    return doc['char_len'] / cpt_by_source[doc['source']] + 1.0


# --- Index every Stage 5 shard --------------------------------------------------------------
MIDTRAIN_REVISION = api.repo_info(MIDTRAIN_REPO, repo_type='dataset').sha
midtrain_shards = sorted(f for f in list_repo_paths(MIDTRAIN_REPO, token=HF_TOKEN)
                         if f.startswith(f'{MIDTRAIN_SRC_PREFIX}/') and f.endswith('.parquet'))
assert midtrain_shards, f"No shards under {MIDTRAIN_REPO}/{MIDTRAIN_SRC_PREFIX}/"

midtrain_index = []
t0 = time.time()
for shard_path in tqdm.tqdm(midtrain_shards, desc="Indexing midtrain"):
    local = robust_hf_download(shard_path, repo_id=MIDTRAIN_REPO, token=HF_TOKEN)
    docs, n_sampled = index_shard(local, shard_path, 'midtrain', len(midtrain_index))
    midtrain_index.extend(docs)

CPT_MIDTRAIN = chars_per_token('midtrain')
MIDTRAIN_CORPUS_CHARS = sum(d['char_len'] for d in midtrain_index)
MIDTRAIN_CORPUS_TOKENS_EST = sum(d['char_len'] / CPT_MIDTRAIN + 1.0 for d in midtrain_index)
print(f"Midtrain corpus: {len(midtrain_index):,} docs / {MIDTRAIN_CORPUS_CHARS:,} chars across "
      f"{len(midtrain_shards)} shard(s) in {fmt_secs(time.time() - t0)}")
print(f"  chars/token = {CPT_MIDTRAIN:.4f} (sampled {CALIBRATION['midtrain']['docs']:,} docs / "
      f"{CALIBRATION['midtrain']['chars']:,} chars)"
      + ("  [CPT_OVERRIDE]" if 'midtrain' in CPT_OVERRIDE else ""))
print(f"  estimated corpus size: {MIDTRAIN_CORPUS_TOKENS_EST:,.0f} tokens")

# --- Epoch pre-flight: fail before touching the old corpus -----------------------------------
# The validation shard's midtrain docs are held out of training, so the epoch budget is measured
# against the corpus minus that reservation -- estimated conservatively here as the full corpus.
epochs_planned = MIDTRAIN_TOKENS_NEEDED / MIDTRAIN_CORPUS_TOKENS_EST
print(f"\nMidtrain need: {MIDTRAIN_TOKENS_NEEDED:,.0f} tokens -> {epochs_planned:.3f} epochs "
      f"(cap {MIDTRAIN_MAX_EPOCHS:.2f})")
if epochs_planned > MIDTRAIN_MAX_EPOCHS:
    raise RuntimeError(
        f"ABORT before downloading the old corpus: this mixture needs {MIDTRAIN_TOKENS_NEEDED:,.0f} "
        f"midtrain tokens from a {MIDTRAIN_CORPUS_TOKENS_EST:,.0f}-token corpus = "
        f"{epochs_planned:.3f} epochs, over the MIDTRAIN_MAX_EPOCHS={MIDTRAIN_MAX_EPOCHS:.2f} cap. "
        f"Raise the cap if that reuse is intended, or lower RATIO_MIDTRAIN / "
        f"EXPECTED_CONSUMED_TOKENS.")
print(f"Epoch check OK: {epochs_planned:.3f} <= {MIDTRAIN_MAX_EPOCHS:.2f}")

## 4. Index the old corpus -- forward from `OLD_START_PCT`, stopping when it has enough

The old corpus is consumed **forward only**: indexing starts at shard
`floor(OLD_START_PCT * n_shards)` and walks upward, skipping `OLD_EXCLUDE_SHARDS`, until the pool
covers `OLD_TOKENS_NEEDED * OLD_INDEX_SLACK` estimated tokens. If it reaches the end of the corpus
first it raises rather than wrapping around -- wrapping would re-enter the base run's pretraining
shards, and no pretraining document in this mixture may have been seen by the base run.

Shard order is *not* shuffled here (that would defeat the forward-only guarantee); documents are
shuffled within a sliding window of shards in section 6, which is what keeps packing streamable.

Each shard is ~228M chars, so this is the download-heavy cell: roughly 11 GB for r30, 3 GB for r60.
Shards land in the HF cache, so section 8 re-parses them without re-downloading.

In [ ]:
OLD_REVISION = api.repo_info(OLD_REPO, repo_type='dataset').sha
old_shards_all = sorted((f for f in list_repo_paths(OLD_REPO) if re.match(r'^shard_\d+\.parquet$', f)),
                        key=shard_index_of)
assert old_shards_all, f"No shards at the root of {OLD_REPO}"

start_pos = int(OLD_START_PCT * len(old_shards_all))
old_shards_window = [s for s in old_shards_all[start_pos:] if shard_index_of(s) not in OLD_EXCLUDE_SHARDS]
assert old_shards_window, f"OLD_START_PCT={OLD_START_PCT} leaves no shards to index."
print(f"Old corpus: {len(old_shards_all)} shard(s) (indices {shard_index_of(old_shards_all[0])}.."
      f"{shard_index_of(old_shards_all[-1])}).")
print(f"  starting {OLD_START_PCT:.0%} in -> position {start_pos} = {old_shards_window[0]}; "
      f"{len(old_shards_window)} shard(s) available forward "
      f"(excluded: {sorted(OLD_EXCLUDE_SHARDS)})")
print(f"  need >= {OLD_TOKENS_NEEDED:,.0f} tokens x {OLD_INDEX_SLACK} slack "
      f"= {OLD_TOKENS_NEEDED * OLD_INDEX_SLACK:,.0f} estimated tokens")

old_index, old_shards_indexed = [], []
pool_chars = 0
stop_at_tokens = OLD_TOKENS_NEEDED * OLD_INDEX_SLACK
t0 = time.time()
progress = tqdm.tqdm(old_shards_window, desc="Indexing old corpus")
for shard_path in progress:
    local = robust_hf_download(shard_path, repo_id=OLD_REPO, token=None)
    docs, _ = index_shard(local, shard_path, 'old_corpus', len(old_index))
    old_index.extend(docs)
    old_shards_indexed.append(shard_path)
    pool_chars += sum(d['char_len'] for d in docs)
    # chars/token only stabilizes once a shard has been sampled, hence the recompute each loop.
    cpt_old_running = chars_per_token('old_corpus')
    pool_tokens = pool_chars / cpt_old_running + len(old_index)
    progress.set_postfix_str(f"{pool_tokens / 1e9:.2f}B/{stop_at_tokens / 1e9:.2f}B tok")
    if pool_tokens >= stop_at_tokens:
        break
progress.close()

CPT_OLD = chars_per_token('old_corpus')
OLD_POOL_TOKENS_EST = pool_chars / CPT_OLD + len(old_index)
print(f"\nOld-corpus pool: {len(old_index):,} docs / {pool_chars:,} chars across "
      f"{len(old_shards_indexed)} shard(s) "
      f"({shard_index_of(old_shards_indexed[0])}..{shard_index_of(old_shards_indexed[-1])}) "
      f"in {fmt_secs(time.time() - t0)}")
print(f"  chars/token = {CPT_OLD:.4f} (sampled {CALIBRATION['old_corpus']['docs']:,} docs / "
      f"{CALIBRATION['old_corpus']['chars']:,} chars)"
      + ("  [CPT_OVERRIDE]" if 'old_corpus' in CPT_OVERRIDE else ""))
print(f"  estimated pool size: {OLD_POOL_TOKENS_EST:,.0f} tokens")

if OLD_POOL_TOKENS_EST < OLD_TOKENS_NEEDED:
    raise RuntimeError(
        f"ABORT: the old corpus ran out. Indexing forward from {OLD_START_PCT:.0%} "
        f"(shard {shard_index_of(old_shards_window[0])}) through the end of the corpus yields only "
        f"{OLD_POOL_TOKENS_EST:,.0f} tokens, but this mixture needs {OLD_TOKENS_NEEDED:,.0f}. "
        f"This notebook never wraps around to shard 0 -- those shards belong to the base run. "
        f"Lower OLD_START_PCT, lower EXPECTED_CONSUMED_TOKENS, or raise RATIO_MIDTRAIN.")

CPT_BY_SOURCE = {'midtrain': CPT_MIDTRAIN, 'old_corpus': CPT_OLD}

## 5. Deduplicate on stable doc ids + normalized-text hashes

Two documents collide if their whitespace-collapsed lowercase text hashes equal. Midtrain is
processed first, so a book present in both corpora is kept as *midtrain* and dropped from the
pretraining pool -- midtrain is the scarce side of the mixture, and this keeps the two components
strictly disjoint in content, not just in provenance.

Counts recorded here (intra-source duplicates, cross-source overlaps) go into the manifest.

In [ ]:
seen_hashes = {}          # hash -> doc_id of the first document that claimed it
DEDUP_STATS = {'midtrain_internal': 0, 'old_internal': 0, 'cross_source': 0,
               'empty_docs_dropped': 0}

midtrain_docs, old_docs = [], []
for pool, keep in ((midtrain_index, midtrain_docs), (old_index, old_docs)):
    for d in pool:
        if d['char_len'] == 0:
            DEDUP_STATS['empty_docs_dropped'] += 1
            continue
        owner = seen_hashes.get(d['hash'])
        if owner is None:
            seen_hashes[d['hash']] = doc_id(d)
            keep.append(d)
        elif owner.startswith(d['source']):
            DEDUP_STATS['midtrain_internal' if d['source'] == 'midtrain' else 'old_internal'] += 1
        else:
            DEDUP_STATS['cross_source'] += 1

MIDTRAIN_UNIQUE_TOKENS_EST = sum(est_tokens(d, CPT_BY_SOURCE) for d in midtrain_docs)
OLD_UNIQUE_TOKENS_EST = sum(est_tokens(d, CPT_BY_SOURCE) for d in old_docs)

print(f"Midtrain: {len(midtrain_docs):,} unique docs "
      f"(-{DEDUP_STATS['midtrain_internal']:,} internal dupes) "
      f"~{MIDTRAIN_UNIQUE_TOKENS_EST:,.0f} tokens")
print(f"Old corpus: {len(old_docs):,} unique docs "
      f"(-{DEDUP_STATS['old_internal']:,} internal dupes, "
      f"-{DEDUP_STATS['cross_source']:,} that duplicate a midtrain doc) "
      f"~{OLD_UNIQUE_TOKENS_EST:,.0f} tokens")
if DEDUP_STATS['empty_docs_dropped']:
    print(f"Dropped {DEDUP_STATS['empty_docs_dropped']:,} empty document(s).")

assert OLD_UNIQUE_TOKENS_EST >= OLD_TOKENS_NEEDED, (
    f"After dedup the pretraining pool holds only {OLD_UNIQUE_TOKENS_EST:,.0f} tokens vs "
    f"{OLD_TOKENS_NEEDED:,.0f} needed. Raise OLD_INDEX_SLACK and rerun section 4.")

## 6. Build the streams, hold out validation, and interleave to the target ratio

**Windowed shuffle.** Each source's documents are shuffled inside a sliding window of
`SHUFFLE_WINDOW_*_SHARDS` source shards rather than globally. A global shuffle would scatter each
output shard's documents across every source shard, which is exactly what forced v1's memory-bound
multi-pass packer; a windowed shuffle gives the same local randomness while letting section 8
stream each source shard through a small LRU and touch it about once. The window size is recorded
in the manifest, and raising it trades RAM for a more global shuffle.

**Validation first.** A held-out shard is built at the target ratio from the head of both streams,
and those documents are then removed from the pools -- including from the cyclic midtrain stream,
so a validation book cannot reappear in training on a later epoch.

**Midtrain is cyclic.** The training stream starts `MIDTRAIN_START_PCT` into the (windowed-shuffled)
midtrain order and wraps around the end, so repeats are spread as far apart as possible. Its epoch
count is re-checked here against `MIDTRAIN_MAX_EPOCHS` after the validation hold-out.

**Ratio-tracking interleave.** Instead of shuffling both sources together and hoping, each step
takes the next midtrain document iff the running midtrain token share is below `RATIO_MIDTRAIN`.
The running ratio therefore stays within one document of target at *every* prefix -- which is what
the requirement "the ratio must hold over the first consumed prefix, not just the whole dataset"
asks for, since the trainer stops partway through the uploaded data.

In [ ]:
def windowed_shuffle(docs, shard_order, window_shards, seed):
    """Shuffle documents inside a sliding window of `window_shards` source shards, keeping the
    windows themselves in `shard_order`. Local randomness, streamable packing."""
    by_shard = defaultdict(list)
    for d in docs:
        by_shard[d['shard_path']].append(d)
    rng = random.Random(seed)
    out = []
    for i in range(0, len(shard_order), window_shards):
        window = [d for sp in shard_order[i:i + window_shards] for d in by_shard.get(sp, [])]
        rng.shuffle(window)
        out.extend(window)
    return out


class SequentialStream:
    """Emits each document once, in order. Returns None when exhausted (never repeats)."""

    def __init__(self, docs):
        self.docs, self.i = docs, 0

    def next(self):
        if self.i >= len(self.docs):
            return None
        d = dict(self.docs[self.i])
        self.i += 1
        d['occurrence'] = 0
        return d


class CyclicStream:
    """Emits documents from `start_frac` through the list and wraps around, so repeats are spread
    a full pass apart. Each emission is a copy tagged with its 0-based occurrence."""

    def __init__(self, docs, start_frac):
        self.docs, self.i = docs, 0
        self.start = int(start_frac * len(docs)) % len(docs) if docs else 0

    def next(self):
        if not self.docs:
            return None
        d = dict(self.docs[(self.start + self.i) % len(self.docs)])
        d['occurrence'] = self.i // len(self.docs)
        self.i += 1
        return d


def interleave(mid_stream, old_stream, target_tokens, ratio):
    """Take documents one at a time, each step from whichever source keeps the running midtrain
    token share on `ratio`. Every prefix of the result is on target to within one document."""
    seq, total, mid_tokens = [], 0.0, 0.0
    while total < target_tokens:
        if ratio >= 1.0:
            want_mid = True
        elif ratio <= 0.0:
            want_mid = False
        else:
            want_mid = (ratio > 0) if total == 0 else (mid_tokens < ratio * total)
        primary, fallback = (mid_stream, old_stream) if want_mid else (old_stream, mid_stream)
        d = primary.next()
        if d is None:
            d = fallback.next()
        if d is None:
            break                      # both sources exhausted
        d['est_tokens'] = est_tokens(d, CPT_BY_SOURCE)
        seq.append(d)
        total += d['est_tokens']
        if d['source'] == 'midtrain':
            mid_tokens += d['est_tokens']
    return seq, total, mid_tokens


# --- Shuffle each source inside its window ---------------------------------------------------
midtrain_shard_order = list(midtrain_shards)
random.Random(RANDOM_SEED).shuffle(midtrain_shard_order)   # midtrain shards have no meaningful order
mid_order = windowed_shuffle(midtrain_docs, midtrain_shard_order,
                             SHUFFLE_WINDOW_MIDTRAIN_SHARDS, RANDOM_SEED + 1)
old_order = windowed_shuffle(old_docs, old_shards_indexed,   # ascending: forward-only stays forward
                             SHUFFLE_WINDOW_OLD_SHARDS, RANDOM_SEED + 2)

# --- Validation shard: taken off the head of both streams, then removed from the pools --------
val_seq, val_tokens_est, val_mid_tokens_est = interleave(
    SequentialStream(mid_order), SequentialStream(old_order), VAL_TARGET_TOKENS, RATIO_MIDTRAIN)
assert val_tokens_est >= VAL_TOKENS, (
    f"Validation shard came out at {val_tokens_est:,.0f} estimated tokens, under the "
    f"{VAL_TOKENS:,} floor. Raise VAL_MARGIN_FRAC.")
val_ids = {doc_id(d) for d in val_seq}
mid_train_pool = [d for d in mid_order if doc_id(d) not in val_ids]
old_train_pool = [d for d in old_order if doc_id(d) not in val_ids]
print(f"Validation: {len(val_seq)} docs, ~{val_tokens_est:,.0f} tokens "
      f"({100 * val_mid_tokens_est / max(val_tokens_est, 1):.1f}% midtrain), held out of training.")

# --- Training sequence ------------------------------------------------------------------------
mid_stream = CyclicStream(mid_train_pool, MIDTRAIN_START_PCT)
old_stream = SequentialStream(old_train_pool)
train_seq, train_tokens_est, train_mid_tokens_est = interleave(
    mid_stream, old_stream, BUILD_TARGET_TOKENS, RATIO_MIDTRAIN)

if train_tokens_est < BUILD_TARGET_TOKENS:
    raise RuntimeError(
        f"ABORT: ran out of source documents at {train_tokens_est:,.0f} of "
        f"{BUILD_TARGET_TOKENS:,} target tokens (old pool exhausted: "
        f"{old_stream.i}/{len(old_train_pool)} docs used). Raise OLD_INDEX_SLACK and rerun "
        f"section 4.")

# --- Epoch check, now against the post-holdout pool ------------------------------------------
MIDTRAIN_POOL_TOKENS_EST = sum(est_tokens(d, CPT_BY_SOURCE) for d in mid_train_pool)
epochs_docs = mid_stream.i / len(mid_train_pool)
epochs_tokens = train_mid_tokens_est / MIDTRAIN_POOL_TOKENS_EST
if max(epochs_docs, epochs_tokens) > MIDTRAIN_MAX_EPOCHS:
    raise RuntimeError(
        f"ABORT before packing: the plan consumes {epochs_tokens:.3f} midtrain epochs by tokens "
        f"({epochs_docs:.3f} by documents), over the MIDTRAIN_MAX_EPOCHS={MIDTRAIN_MAX_EPOCHS:.2f} "
        f"cap. Raise the cap if that reuse is intended, or lower RATIO_MIDTRAIN / "
        f"EXPECTED_CONSUMED_TOKENS.")

# --- Plan output shards (deterministic, index-only) -------------------------------------------
plan, cur, cur_tokens = [], [], 0.0
for d in train_seq:
    cur.append(d)
    cur_tokens += d['est_tokens']
    if cur_tokens >= TOKENS_PER_SHARD:
        plan.append(cur)
        cur, cur_tokens = [], 0.0
if cur:
    plan.append(cur)

VAL_SHARD_IDX = len(plan)          # sorts after every train shard, as nanochat expects
all_shards = plan + [val_seq]
for k, docs in enumerate(all_shards):
    for pos, d in enumerate(docs):
        d['out_idx'], d['out_pos'] = k, pos

# Which old-corpus shards this mixture actually draws from -- the disjointness check in section 7
# and the manifest both use this.
OLD_SHARDS_USED = sorted({shard_index_of(d['shard_path'])
                          for docs in all_shards for d in docs if d['source'] == 'old_corpus'})
MIDTRAIN_SHARDS_USED = sorted({shard_index_of(d['shard_path'])
                               for docs in all_shards for d in docs if d['source'] == 'midtrain'})

print(f"\nTraining sequence: {len(train_seq):,} docs, ~{train_tokens_est:,.0f} tokens "
      f"({100 * train_mid_tokens_est / train_tokens_est:.2f}% midtrain estimated)")
print(f"Midtrain reuse: {mid_stream.i:,} emissions from a pool of {len(mid_train_pool):,} docs "
      f"= {epochs_docs:.3f} epochs by docs / {epochs_tokens:.3f} by tokens "
      f"(cap {MIDTRAIN_MAX_EPOCHS:.2f}, start {MIDTRAIN_START_PCT:.0%})")
print(f"Pretraining draw: {old_stream.i:,} unique docs, no repeats, from {len(OLD_SHARDS_USED)} "
      f"old shard(s) {OLD_SHARDS_USED[0]}..{OLD_SHARDS_USED[-1]}")
print(f"Planned {len(plan)} train shard(s) + validation shard_{VAL_SHARD_IDX:05d}.parquet")

## 7. Pre-flight: prefix ratio, source disjointness, shard preview

Everything checkable before a byte is uploaded:

- the **estimated ratio over the consumed prefix** (the first `EXPECTED_CONSUMED_TOKENS`), which is
  the number the run actually experiences -- section 9 re-measures it exactly;
- no pretraining document comes from the base run's shards `0..BASE_RUN_OLD_SHARDS_UPTO`;
- no old-corpus shard used here is already used by another `mixed/v2/*` mixture, so the r30 and r60
  pretraining portions stay disjoint (set `ENFORCE_DISJOINT_OLD_RANGE = False` to downgrade to a
  warning).

In [ ]:
def prefix_ratio(seq, limit_tokens, token_key):
    """Midtrain share of the first `limit_tokens` of a document sequence, counting a document that
    straddles the boundary only up to the boundary (the trainer stops mid-document too)."""
    total = mid = 0.0
    for d in seq:
        t = d[token_key]
        if total + t > limit_tokens:
            t = limit_tokens - total
            if d['source'] == 'midtrain':
                mid += t
            total += t
            break
        if d['source'] == 'midtrain':
            mid += t
        total += t
    return (100.0 * mid / total if total else 0.0), total


consumed_pct_est, consumed_tokens_est = prefix_ratio(train_seq, EXPECTED_CONSUMED_TOKENS, 'est_tokens')
whole_pct_est = 100.0 * train_mid_tokens_est / train_tokens_est
print(f"Estimated midtrain share over the consumed prefix ({consumed_tokens_est:,.0f} tokens): "
      f"{consumed_pct_est:.3f}%  (target {100 * RATIO_MIDTRAIN:.3f}%)")
print(f"Estimated midtrain share over the whole upload: {whole_pct_est:.3f}%")
print("(Both are estimates from the chars/token calibration; section 9 measures them exactly.)")

# --- No pretraining document may come from the base run's shards ------------------------------
base_overlap = [s for s in OLD_SHARDS_USED if s <= BASE_RUN_OLD_SHARDS_UPTO]
if base_overlap:
    raise RuntimeError(
        f"ABORT: {len(base_overlap)} old-corpus shard(s) in this plan (e.g. {base_overlap[:5]}) are "
        f"within the base run's pretraining range 0..{BASE_RUN_OLD_SHARDS_UPTO}. Raise "
        f"OLD_START_PCT.")
print(f"\nBase-run overlap: none (uses old shards {OLD_SHARDS_USED[0]}..{OLD_SHARDS_USED[-1]}, all "
      f"> {BASE_RUN_OLD_SHARDS_UPTO}).")

# --- No other v2 mixture may already be drawing from these old shards -------------------------
sibling_manifests = sorted(f for f in list_repo_paths(MIDTRAIN_REPO, token=HF_TOKEN)
                           if f.startswith('mixed/v2/') and f.endswith('/manifest.json')
                           and not f.startswith(f'{DST_PREFIX}/'))
collisions = {}
for path in sibling_manifests:
    other = json.load(open(robust_hf_download(path, repo_id=MIDTRAIN_REPO, token=HF_TOKEN)))
    other_shards = set(other.get('sources', {}).get('old_corpus_shards_used', []))
    shared = sorted(other_shards & set(OLD_SHARDS_USED))
    if shared:
        collisions[path] = shared
if collisions:
    detail = '; '.join(f"{p}: {len(v)} shard(s) {v[:5]}{'...' if len(v) > 5 else ''}"
                       for p, v in collisions.items())
    msg = (f"Old-corpus shards in this plan are already used by another v2 mixture -- the "
           f"pretraining portions would not be disjoint. {detail}")
    if ENFORCE_DISJOINT_OLD_RANGE:
        raise RuntimeError(f"ABORT: {msg} Move OLD_START_PCT past them, or set "
                           f"ENFORCE_DISJOINT_OLD_RANGE = False if the overlap is intended.")
    print(f"WARNING: {msg}")
else:
    print(f"Disjointness: no old-corpus shard is shared with the {len(sibling_manifests)} other "
          f"mixed/v2 mixture(s) on the repo.")

# --- Packing fan-out: how many source shards feed one output shard ----------------------------
# This is what SOURCE_CACHE_SHARDS has to cover for section 8 to parse each source shard about
# once. A cache miss only costs a re-parse of a locally cached file, so exceeding it is slow,
# not wrong.
fanout = max(len({(d['source'], d['shard_path']) for d in docs}) for docs in all_shards)
print(f"\nPacking fan-out: at most {fanout} source shard(s) feed one output shard "
      f"(SOURCE_CACHE_SHARDS={SOURCE_CACHE_SHARDS}).")
if fanout > SOURCE_CACHE_SHARDS:
    print(f"  NOTE: raise SOURCE_CACHE_SHARDS to >= {fanout} (~{fanout * 0.2:.1f} GB RAM) to avoid "
          f"re-parsing source shards, or lower the SHUFFLE_WINDOW_* sizes.")

# --- Shard preview -----------------------------------------------------------------------------
print(f"\n{len(plan)} train shard(s) + 1 validation shard:")
for k in list(range(min(3, len(plan)))) + ([len(plan) - 1] if len(plan) > 3 else []):
    docs = all_shards[k]
    n_mid = sum(1 for d in docs if d['source'] == 'midtrain')
    tok = sum(d['est_tokens'] for d in docs)
    mid_tok = sum(d['est_tokens'] for d in docs if d['source'] == 'midtrain')
    print(f"  shard_{k:05d}: {len(docs):>5,} docs ({n_mid:>4,} midtrain) | ~{tok:>13,.0f} tokens "
          f"| {100 * mid_tok / tok:5.2f}% midtrain")
    if k == 2 and len(plan) > 4:
        print(f"  ...")
val_tok = sum(d['est_tokens'] for d in val_seq)
val_mid = sum(d['est_tokens'] for d in val_seq if d['source'] == 'midtrain')
print(f"  shard_{VAL_SHARD_IDX:05d}: {len(val_seq):>5,} docs "
      f"({sum(1 for d in val_seq if d['source'] == 'midtrain'):>4,} midtrain) | "
      f"~{val_tok:>13,.0f} tokens | {100 * val_mid / val_tok:5.2f}% midtrain  [VALIDATION]")

## 8. Pack + upload -- streaming, exact-tokenized, resumable

Output shards are built **in order**, each one gathering its documents from an LRU of parsed source
shards. Because both streams are windowed (section 6), a shard's documents come from a handful of
consecutive source shards, so each source shard is parsed roughly once -- no multi-pass re-reading,
and peak RAM is `SOURCE_CACHE_SHARDS` source shards (~230MB each) plus the one output shard being
assembled.

Every document is **exact-tokenized as it is written**. That is the "verify" half of the
estimate-then-verify design: the manifest's token counts and ratios are measured, not projected.
Per-shard sidecars (`shard_meta/shard_NNNNN.json`) record every document's id, source, occurrence,
chars, exact tokens and text hash, plus the shard's sha256 -- so a resumed run can verify and
report on shards it did not rebuild, without re-downloading them.

Shards upload in batches of `SHARDS_PER_COMMIT` (one HF commit each, parquet + sidecar together),
and shards already present are skipped. Before trusting that skip, a guard reads back a sample of
the existing shards and compares their per-row character lengths against the current plan: a
folder built with different parameters aborts here instead of being welded onto this mixture.

In [ ]:
class SourceShardCache:
    """LRU of parsed source-shard text columns. Downloads are HF-cached on disk, so an eviction
    costs a re-parse, not a re-download."""

    def __init__(self, capacity):
        self.capacity, self.store = capacity, OrderedDict()
        self.hits = self.misses = 0

    def texts(self, source, shard_path):
        key = (source, shard_path)
        if key in self.store:
            self.hits += 1
            self.store.move_to_end(key)
            return self.store[key]
        self.misses += 1
        repo_id = MIDTRAIN_REPO if source == 'midtrain' else OLD_REPO
        token = HF_TOKEN if source == 'midtrain' else None
        col = read_text_column(robust_hf_download(shard_path, repo_id=repo_id, token=token))
        self.store[key] = col
        while len(self.store) > self.capacity:
            self.store.popitem(last=False)
        return col


def shard_name(k):
    return f'shard_{k:05d}.parquet'


existing_out = {os.path.basename(f) for f in list_repo_paths(MIDTRAIN_REPO, token=HF_TOKEN)
                if f.startswith(f'{DATA_PREFIX}/') and f.endswith('.parquet')}
todo = [k for k in range(len(all_shards)) if shard_name(k) not in existing_out]
print(f"Shards: {len(all_shards)} planned ({len(plan)} train + 1 validation) | "
      f"already uploaded: {len(all_shards) - len(todo)} | remaining: {len(todo)}")

# --- Guard: existing shards must belong to THIS plan -------------------------------------------
existing_ks = sorted(int(m.group(1)) for f in existing_out if (m := re.match(r'shard_(\d+)\.parquet$', f)))
orphans = [k for k in existing_ks if k >= len(all_shards)]
if orphans:
    raise RuntimeError(
        f"ABORT: {len(orphans)} existing shard(s) (e.g. {shard_name(orphans[0])}) lie beyond this "
        f"plan of {len(all_shards)} shard(s) -- the folder was built with different parameters. "
        f"Wipe {DST_PREFIX}/ (section 1b) and rebuild, or restore the parameters it was built with.")

already_ks = [k for k in existing_ks if k < len(all_shards)]
if already_ks:
    for k in sorted({already_ks[0], already_ks[len(already_ks) // 2], already_ks[-1]}):
        local = robust_hf_download(f'{DATA_PREFIX}/{shard_name(k)}', repo_id=MIDTRAIN_REPO, token=HF_TOKEN)
        got = [len(t) if isinstance(t, str) else 0 for t in read_text_column(local)]
        want = [d['char_len'] for d in all_shards[k]]
        if got != want:
            raise RuntimeError(
                f"ABORT: existing {shard_name(k)} ({len(got)} docs) does not match this plan "
                f"({len(want)} docs). The folder was built with different RANDOM_SEED / ratio / "
                f"token targets / start percentages. Restore those parameters to resume, or wipe "
                f"{DST_PREFIX}/ (section 1b) and rebuild.")
    print("Resume guard OK: the existing shards match this plan.")

# --- Pack ---------------------------------------------------------------------------------------
SHARD_STATS = {}          # k -> sidecar dict for shards written by THIS run
RUN_COMMITS = []          # immutable commit SHAs produced by this run
cache = SourceShardCache(SOURCE_CACHE_SHARDS)
pack_tmp = tempfile.mkdtemp()
pending = []              # [(shard_name, [(path_in_repo, local_path), ...]), ...]
n_committed = 0


def commit_pending(force=False):
    """Commit buffered shards (parquet + sidecar) as ONE HF commit once SHARDS_PER_COMMIT are
    ready, or force=True to flush the tail."""
    global n_committed
    if not pending or (len(pending) < SHARDS_PER_COMMIT and not force):
        return
    ops = [CommitOperationAdd(path_in_repo=pir, path_or_fileobj=loc)
           for _, files in pending for pir, loc in files]
    names = [n for n, _ in pending]
    msg = (f'v2/ratio_{RATIO_PCT}: {names[0]}..{names[-1]} ({len(pending)} shards)'
           if len(pending) > 1 else f'v2/ratio_{RATIO_PCT} {names[0]}')
    info = robust_commit(ops, msg)
    RUN_COMMITS.append(info.oid)
    for _, files in pending:
        for _, loc in files:
            os.remove(loc)
    n_committed += len(pending)
    tqdm.tqdm.write(f"  committed {len(pending)} shard(s) [{names[0]}..{names[-1]}] "
                    f"({n_committed}/{len(todo)} uploaded, commit {info.oid[:10]})")
    pending.clear()


def build_shard(k):
    """Read this shard's documents, exact-tokenize them, write the parquet + sidecar, queue both."""
    docs = all_shards[k]
    texts = [cache.texts(d['source'], d['shard_path'])[d['row_idx']] for d in docs]
    texts = [t if isinstance(t, str) else '' for t in texts]   # null rows never survive dedup
    tokens = token_counts(texts)

    name = shard_name(k)
    local_parquet = os.path.join(pack_tmp, name)
    pq.write_table(pa.table({'text': texts}), local_parquet)
    del texts

    counts = defaultdict(int)
    for d in docs:
        counts[f"{d['source']}:{d['shard_path']}"] += 1
    n_mid = sum(1 for d in docs if d['source'] == 'midtrain')
    sidecar = {
        'shard': name,
        'shard_index': k,
        'split': 'validation' if k == VAL_SHARD_IDX else 'train',
        'n_docs': len(docs),
        'n_midtrain': n_mid,
        'n_old_corpus': len(docs) - n_mid,
        'chars': sum(d['char_len'] for d in docs),
        'tokens': sum(tokens),
        'tokens_midtrain': sum(t for d, t in zip(docs, tokens) if d['source'] == 'midtrain'),
        'tokens_old_corpus': sum(t for d, t in zip(docs, tokens) if d['source'] == 'old_corpus'),
        'sha256': sha256_file(local_parquet),
        'source_shards': dict(sorted(counts.items())),
        'docs': [{'i': d['out_pos'], 'id': doc_id(d), 'src': d['source'], 'occ': d['occurrence'],
                  'chars': d['char_len'], 'tokens': t, 'hash': d['hash']}
                 for d, t in zip(docs, tokens)],
    }
    SHARD_STATS[k] = sidecar
    local_meta = os.path.join(pack_tmp, f'{name}.json')
    with open(local_meta, 'w') as f:
        json.dump(sidecar, f)
    pending.append((name, [(f'{DATA_PREFIX}/{name}', local_parquet),
                           (f'{SHARDMETA_PREFIX}/{name}.json', local_meta)]))
    commit_pending()
    return sidecar


t0 = time.time()
for k in tqdm.tqdm(todo, desc="Packing"):
    s = build_shard(k)
    tqdm.tqdm.write(f"  {s['shard']}: {s['n_docs']:,} docs | {s['tokens']:,} tokens | "
                    f"{100 * s['tokens_midtrain'] / max(s['tokens'], 1):.2f}% midtrain")
commit_pending(force=True)

print(f"\nPacked + committed {n_committed} shard(s) in {fmt_secs(time.time() - t0)} to "
      f"{MIDTRAIN_REPO}/{DATA_PREFIX}/")
print(f"Source-shard cache: {cache.hits:,} hits / {cache.misses:,} misses "
      f"(a miss re-parses a locally cached file)")
if RUN_COMMITS:
    print(f"Data commits this run: {len(RUN_COMMITS)} (latest {RUN_COMMITS[-1]})")

## 9. Verify what was actually written

Reads the per-shard sidecars (rebuilding any that are missing from the uploaded parquet) and
reports **measured** numbers: exact token totals, the midtrain share over the consumed prefix and
over the whole upload, the exact midtrain epoch count, unique-document counts, duplicate and
cross-source overlap counts, and train/validation separation.

If the achieved ratio misses `RATIO_TOLERANCE_PP`, this cell raises and prints the exact chars/token
constants measured from the written data -- pin them in `CPT_OVERRIDE`, wipe (section 1b), and
rebuild; the second pass lands on target because the planner is then working from measured
constants rather than a sample.

In [ ]:
def rebuild_sidecar(k):
    """Re-derive a shard's exact stats from the uploaded parquet (used when resuming a run whose
    sidecar is missing). Cross-checks the shard against the plan before trusting it."""
    local = robust_hf_download(f'{DATA_PREFIX}/{shard_name(k)}', repo_id=MIDTRAIN_REPO, token=HF_TOKEN)
    texts = read_text_column(local)
    docs = all_shards[k]
    assert [len(t) for t in texts] == [d['char_len'] for d in docs], (
        f"{shard_name(k)} on the repo does not match the plan; wipe and rebuild.")
    tokens = token_counts(texts)
    del texts
    counts = defaultdict(int)
    for d in docs:
        counts[f"{d['source']}:{d['shard_path']}"] += 1
    n_mid = sum(1 for d in docs if d['source'] == 'midtrain')
    return {
        'shard': shard_name(k), 'shard_index': k,
        'split': 'validation' if k == VAL_SHARD_IDX else 'train',
        'n_docs': len(docs), 'n_midtrain': n_mid, 'n_old_corpus': len(docs) - n_mid,
        'chars': sum(d['char_len'] for d in docs), 'tokens': sum(tokens),
        'tokens_midtrain': sum(t for d, t in zip(docs, tokens) if d['source'] == 'midtrain'),
        'tokens_old_corpus': sum(t for d, t in zip(docs, tokens) if d['source'] == 'old_corpus'),
        'sha256': sha256_file(local), 'source_shards': dict(sorted(counts.items())),
        'docs': [{'i': d['out_pos'], 'id': doc_id(d), 'src': d['source'], 'occ': d['occurrence'],
                  'chars': d['char_len'], 'tokens': t, 'hash': d['hash']}
                 for d, t in zip(docs, tokens)],
    }


# --- Fill in stats for shards this run skipped ------------------------------------------------
repo_meta = {os.path.basename(f) for f in list_repo_paths(MIDTRAIN_REPO, token=HF_TOKEN)
             if f.startswith(f'{SHARDMETA_PREFIX}/')}
recovered_ops, meta_tmp = [], tempfile.mkdtemp()
for k in range(len(all_shards)):
    if k in SHARD_STATS:
        continue
    if f'{shard_name(k)}.json' in repo_meta:
        SHARD_STATS[k] = json.load(open(robust_hf_download(
            f'{SHARDMETA_PREFIX}/{shard_name(k)}.json', repo_id=MIDTRAIN_REPO, token=HF_TOKEN)))
    else:
        print(f"  {shard_name(k)}: no sidecar on the repo, re-measuring it from the uploaded shard")
        SHARD_STATS[k] = rebuild_sidecar(k)
        local_meta = os.path.join(meta_tmp, f'{shard_name(k)}.json')
        with open(local_meta, 'w') as f:
            json.dump(SHARD_STATS[k], f)
        recovered_ops.append(CommitOperationAdd(
            path_in_repo=f'{SHARDMETA_PREFIX}/{shard_name(k)}.json', path_or_fileobj=local_meta))
if recovered_ops:
    RUN_COMMITS.append(robust_commit(recovered_ops, f'v2/ratio_{RATIO_PCT} recovered sidecars').oid)
    print(f"Uploaded {len(recovered_ops)} recovered sidecar(s).")

# --- Exact totals -------------------------------------------------------------------------------
train_stats = [SHARD_STATS[k] for k in range(len(plan))]
val_stats = SHARD_STATS[VAL_SHARD_IDX]
TRAIN_TOKENS = sum(s['tokens'] for s in train_stats)
TRAIN_TOKENS_MID = sum(s['tokens_midtrain'] for s in train_stats)
TRAIN_TOKENS_OLD = sum(s['tokens_old_corpus'] for s in train_stats)
TRAIN_CHARS = sum(s['chars'] for s in train_stats)
TRAIN_DOCS = sum(s['n_docs'] for s in train_stats)

train_doc_entries = [e for s in train_stats for e in s['docs']]
flat = [{'source': e['src'], 'tok': e['tokens']} for e in train_doc_entries]
ACHIEVED_PCT_PREFIX, PREFIX_TOKENS = prefix_ratio(flat, EXPECTED_CONSUMED_TOKENS, 'tok')
ACHIEVED_PCT_WHOLE = 100.0 * TRAIN_TOKENS_MID / TRAIN_TOKENS

print(f"Exact train totals: {TRAIN_TOKENS:,} tokens / {TRAIN_DOCS:,} docs / {TRAIN_CHARS:,} chars "
      f"across {len(train_stats)} shard(s)")
print(f"  midtrain {TRAIN_TOKENS_MID:,} | pretraining {TRAIN_TOKENS_OLD:,}")
print(f"  supply vs the run: needs {EXPECTED_CONSUMED_TOKENS:,} + {CACHE_MARGIN_FRAC:.0%} margin = "
      f"{BUILD_TARGET_TOKENS:,} -> {'OK' if TRAIN_TOKENS >= BUILD_TARGET_TOKENS else 'SHORT'} "
      f"({TRAIN_TOKENS / EXPECTED_CONSUMED_TOKENS:.4f}x the consumed amount)")
print(f"Validation shard_{VAL_SHARD_IDX:05d}: {val_stats['tokens']:,} tokens / "
      f"{val_stats['n_docs']} docs -> "
      f"{'OK' if val_stats['tokens'] >= VAL_TOKENS else 'SHORT'} vs the {VAL_TOKENS:,} floor")

print(f"\nAchieved midtrain share (exact tokens, BOS included):")
print(f"  over the consumed prefix ({PREFIX_TOKENS:,.0f} tokens): {ACHIEVED_PCT_PREFIX:.4f}%  "
      f"(target {100 * RATIO_MIDTRAIN:.4f}%, delta {ACHIEVED_PCT_PREFIX - 100 * RATIO_MIDTRAIN:+.4f} pp)")
print(f"  over the whole upload ({TRAIN_TOKENS:,} tokens):        {ACHIEVED_PCT_WHOLE:.4f}%  "
      f"(delta {ACHIEVED_PCT_WHOLE - 100 * RATIO_MIDTRAIN:+.4f} pp)")

# --- Reuse, uniqueness, separation --------------------------------------------------------------
mid_entries = [e for e in train_doc_entries if e['src'] == 'midtrain']
old_entries = [e for e in train_doc_entries if e['src'] == 'old_corpus']
exact_tokens_by_id = {e['id']: e['tokens'] for e in mid_entries}
MIDTRAIN_POOL_TOKENS_EXACT = sum(
    exact_tokens_by_id.get(doc_id(d), est_tokens(d, CPT_BY_SOURCE)) for d in mid_train_pool)
measured_share = 100.0 * sum(1 for d in mid_train_pool if doc_id(d) in exact_tokens_by_id) / len(mid_train_pool)
MIDTRAIN_EPOCHS_TOKENS = TRAIN_TOKENS_MID / MIDTRAIN_POOL_TOKENS_EXACT
MIDTRAIN_EPOCHS_DOCS = len(mid_entries) / len(mid_train_pool)

old_ids = [e['id'] for e in old_entries]
old_unique = set(old_ids)
old_hashes = [e['hash'] for e in old_entries]
OLD_REPEATS = len(old_ids) - len(old_unique)
OLD_HASH_DUPES = len(old_hashes) - len(set(old_hashes))
val_ids_written = {e['id'] for e in val_stats['docs']}
val_hashes_written = {e['hash'] for e in val_stats['docs']}
TRAIN_VAL_ID_OVERLAP = len(val_ids_written & {e['id'] for e in train_doc_entries})
TRAIN_VAL_HASH_OVERLAP = len(val_hashes_written & {e['hash'] for e in train_doc_entries})
CROSS_SOURCE_HASH_OVERLAP = len({e['hash'] for e in mid_entries} & set(old_hashes))

print(f"\nMidtrain reuse: {len(mid_entries):,} emissions of {len(set(e['id'] for e in mid_entries)):,} "
      f"unique docs = {MIDTRAIN_EPOCHS_DOCS:.4f} epochs by docs / {MIDTRAIN_EPOCHS_TOKENS:.4f} by "
      f"tokens (pool token count is {measured_share:.1f}% measured, rest estimated)")
print(f"Pretraining: {len(old_unique):,} unique docs, {OLD_REPEATS} repeated id(s), "
      f"{OLD_HASH_DUPES} duplicate text hash(es)")
print(f"Cross-source text overlap (midtrain vs pretraining): {CROSS_SOURCE_HASH_OVERLAP}")
print(f"Validation vs training overlap: {TRAIN_VAL_ID_OVERLAP} doc id(s), "
      f"{TRAIN_VAL_HASH_OVERLAP} text hash(es)")

assert OLD_REPEATS == 0 and OLD_HASH_DUPES == 0, "A pretraining document repeats -- must not happen."
assert TRAIN_VAL_ID_OVERLAP == 0 and TRAIN_VAL_HASH_OVERLAP == 0, "Validation leaked into training."
assert CROSS_SOURCE_HASH_OVERLAP == 0, "A document appears in both the midtrain and pretraining side."
assert MIDTRAIN_EPOCHS_TOKENS <= MIDTRAIN_MAX_EPOCHS, (
    f"Measured midtrain reuse {MIDTRAIN_EPOCHS_TOKENS:.4f} exceeds MIDTRAIN_MAX_EPOCHS.")

# --- Ratio tolerance ------------------------------------------------------------------------------
CPT_MEASURED = {
    'midtrain': sum(e['chars'] for e in mid_entries) / max(sum(e['tokens'] for e in mid_entries) - len(mid_entries), 1),
    'old_corpus': sum(e['chars'] for e in old_entries) / max(sum(e['tokens'] for e in old_entries) - len(old_entries), 1),
}
delta_pp = abs(ACHIEVED_PCT_PREFIX - 100 * RATIO_MIDTRAIN)
if delta_pp > RATIO_TOLERANCE_PP:
    raise RuntimeError(
        f"Achieved {ACHIEVED_PCT_PREFIX:.4f}% midtrain over the consumed prefix, "
        f"{delta_pp:.4f} pp off the {100 * RATIO_MIDTRAIN:.2f}% target (tolerance "
        f"{RATIO_TOLERANCE_PP} pp). The planner's chars/token estimate was off. Set\n"
        f"    CPT_OVERRIDE = {{'midtrain': {CPT_MEASURED['midtrain']:.4f}, "
        f"'old_corpus': {CPT_MEASURED['old_corpus']:.4f}}}\n"
        f"(planned with midtrain={CPT_MIDTRAIN:.4f}, old_corpus={CPT_OLD:.4f}), wipe the folder "
        f"with section 1b, and rerun -- the second pass plans from measured constants.")
print(f"\nRatio OK: {delta_pp:.4f} pp off target, within the {RATIO_TOLERANCE_PP} pp tolerance.")
print(f"Measured chars/token: midtrain {CPT_MEASURED['midtrain']:.4f} (planned {CPT_MIDTRAIN:.4f}) | "
      f"old_corpus {CPT_MEASURED['old_corpus']:.4f} (planned {CPT_OLD:.4f})")

## 10. Manifest + README, and the immutable commit SHA to hand back

`manifest.json` records everything needed to audit or reproduce the mixture: generator commit,
source repos with their immutable revisions, tokenizer fingerprint, seed, shard counts and the
validation index, exact token totals, achieved ratios over the consumed prefix, unique-document
counts, duplicate/overlap counts, and a sha256 for every shard.

The commit SHA printed at the end is the immutable revision to quote when handing the dataset over
-- never `main`.

In [ ]:
shard_records = [
    {k: SHARD_STATS[i][k] for k in
     ('shard', 'shard_index', 'split', 'n_docs', 'n_midtrain', 'n_old_corpus',
      'chars', 'tokens', 'tokens_midtrain', 'tokens_old_corpus', 'sha256', 'source_shards')}
    for i in range(len(all_shards))
]

manifest = {
    'dataset': f'{MIDTRAIN_REPO}/{DST_PREFIX}',
    'supersedes': f'{MIDTRAIN_REPO}/mixed/ratio_{RATIO_PCT}',
    'generator': {
        'notebook': GENERATOR_NOTEBOOK,
        'repo': GENERATOR_CODE_REPO,
        'code_commit': GENERATOR_CODE_COMMIT,
    },
    'sources': {
        'midtrain': {'repo': MIDTRAIN_REPO, 'prefix': MIDTRAIN_SRC_PREFIX, 'revision': MIDTRAIN_REVISION},
        'old_corpus': {'repo': OLD_REPO, 'revision': OLD_REVISION},
        'old_corpus_shards_used': OLD_SHARDS_USED,
        'old_corpus_shards_indexed': [shard_index_of(s) for s in old_shards_indexed],
        'old_corpus_start_pct': OLD_START_PCT,
        'old_corpus_excluded_shards': sorted(OLD_EXCLUDE_SHARDS),
        'base_run_old_shards_upto': BASE_RUN_OLD_SHARDS_UPTO,
        'midtrain_shards_used': MIDTRAIN_SHARDS_USED,
        'midtrain_start_pct': MIDTRAIN_START_PCT,
    },
    'tokenizer': {
        'repo': TOKENIZER_REPO, 'prefix': TOKENIZER_PREFIX, 'name': enc.name,
        'vocab_size': VOCAB_SIZE, 'bos_token_id': BOS_ID, 'bos_counted_per_document': True,
        'fingerprint_sha256': TOKENIZER_FINGERPRINT,
    },
    'shuffle': {
        'seed': RANDOM_SEED,
        'scheme': 'windowed shuffle within a sliding window of source shards, then a '
                  'ratio-tracking interleave of the two sources',
        'window_shards_old_corpus': SHUFFLE_WINDOW_OLD_SHARDS,
        'window_shards_midtrain': SHUFFLE_WINDOW_MIDTRAIN_SHARDS,
    },
    'layout': {
        'data_prefix': DATA_PREFIX,
        'shard_meta_prefix': SHARDMETA_PREFIX,
        'n_train_shards': len(plan),
        'validation_shard_index': VAL_SHARD_IDX,
        'tokens_per_shard_target': TOKENS_PER_SHARD,
        'columns': ['text'],
    },
    'targets': {
        'ratio_midtrain': RATIO_MIDTRAIN,
        'expected_consumed_tokens': EXPECTED_CONSUMED_TOKENS,
        'cache_margin_frac': CACHE_MARGIN_FRAC,
        'build_target_tokens': BUILD_TARGET_TOKENS,
        'val_tokens_floor': VAL_TOKENS,
        'midtrain_max_epochs': MIDTRAIN_MAX_EPOCHS,
        'ratio_tolerance_pp': RATIO_TOLERANCE_PP,
    },
    'tokens': {
        'train_total': TRAIN_TOKENS,
        'train_midtrain': TRAIN_TOKENS_MID,
        'train_pretraining': TRAIN_TOKENS_OLD,
        'validation_total': val_stats['tokens'],
        'validation_midtrain': val_stats['tokens_midtrain'],
        'validation_pretraining': val_stats['tokens_old_corpus'],
        'consumed_prefix_tokens': int(PREFIX_TOKENS),
    },
    'achieved': {
        'midtrain_pct_consumed_prefix': round(ACHIEVED_PCT_PREFIX, 4),
        'midtrain_pct_whole_upload': round(ACHIEVED_PCT_WHOLE, 4),
        'delta_pp_vs_target': round(ACHIEVED_PCT_PREFIX - 100 * RATIO_MIDTRAIN, 4),
        'midtrain_epochs_by_tokens': round(MIDTRAIN_EPOCHS_TOKENS, 4),
        'midtrain_epochs_by_docs': round(MIDTRAIN_EPOCHS_DOCS, 4),
        'chars_per_token_measured': {k: round(v, 4) for k, v in CPT_MEASURED.items()},
        'chars_per_token_planned': {'midtrain': round(CPT_MIDTRAIN, 4), 'old_corpus': round(CPT_OLD, 4)},
    },
    'documents': {
        'train_documents': TRAIN_DOCS,
        'train_unique_documents': len({e['id'] for e in train_doc_entries}),
        'midtrain_emissions': len(mid_entries),
        'midtrain_unique_documents': len({e['id'] for e in mid_entries}),
        'midtrain_pool_documents': len(mid_train_pool),
        'pretraining_unique_documents': len(old_unique),
        'validation_documents': val_stats['n_docs'],
    },
    'dedup': {
        'index_midtrain_internal_duplicates': DEDUP_STATS['midtrain_internal'],
        'index_old_corpus_internal_duplicates': DEDUP_STATS['old_internal'],
        'index_cross_source_duplicates': DEDUP_STATS['cross_source'],
        'index_empty_documents_dropped': DEDUP_STATS['empty_docs_dropped'],
        'written_pretraining_repeated_ids': OLD_REPEATS,
        'written_pretraining_duplicate_hashes': OLD_HASH_DUPES,
        'written_cross_source_hash_overlap': CROSS_SOURCE_HASH_OVERLAP,
        'written_train_validation_id_overlap': TRAIN_VAL_ID_OVERLAP,
        'written_train_validation_hash_overlap': TRAIN_VAL_HASH_OVERLAP,
        'method': 'stable document ids (source:shard#row) + sha1 of lowercased, '
                  'whitespace-collapsed text',
    },
    'shard_records': shard_records,
    'updated_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}

meta_out = tempfile.mkdtemp()
manifest_local = os.path.join(meta_out, 'manifest.json')
with open(manifest_local, 'w') as f:
    json.dump(manifest, f, indent=2, sort_keys=True)

shard_table = '\n'.join(
    f"| `{r['shard']}` | {r['split']} | {r['n_docs']:,} | {r['tokens']:,} | "
    f"{100 * r['tokens_midtrain'] / max(r['tokens'], 1):.2f}% | `{r['sha256'][:16]}...` |"
    for r in shard_records)

# Hoisted out of the README f-string below (which is triple-quoted; keeping subscripts out of it
# avoids quote-nesting rules that differ across Python versions).
val_tokens_written = val_stats['tokens']
val_docs_written = val_stats['n_docs']
val_pct_midtrain = 100 * val_stats['tokens_midtrain'] / max(val_stats['tokens'], 1)
mid_unique_written = len({e['id'] for e in mid_entries})
dedup_old_internal = DEDUP_STATS['old_internal']
dedup_cross_source = DEDUP_STATS['cross_source']
updated_at = manifest['updated_at']

readme = f'''# Mixed corpus v2 -- ratio_{RATIO_PCT} ({RATIO_PCT}% midtrain by tokens)

Blend of [`{MIDTRAIN_REPO}/{MIDTRAIN_SRC_PREFIX}`](https://huggingface.co/datasets/{MIDTRAIN_REPO}/tree/main/{MIDTRAIN_SRC_PREFIX})
(midtrain) and [`{OLD_REPO}`](https://huggingface.co/datasets/{OLD_REPO}/tree/main) (original
pretraining corpus), produced by `{GENERATOR_NOTEBOOK}`. Text-only shards (`text` column),
document boundaries preserved -- nothing is concatenated or split to force the ratio.

**This supersedes `mixed/ratio_{RATIO_PCT}`**, which targeted its ratio by document count and
therefore under-delivered midtrain content by token. That folder is left untouched.

## Composition (measured with the run's exact tokenizer, BOS counted per document)

| | tokens | share |
|---|---|---|
| midtrain | {TRAIN_TOKENS_MID:,} | {ACHIEVED_PCT_WHOLE:.4f}% |
| pretraining | {TRAIN_TOKENS_OLD:,} | {100 - ACHIEVED_PCT_WHOLE:.4f}% |
| **train total** | **{TRAIN_TOKENS:,}** | |
| validation (`shard_{VAL_SHARD_IDX:05d}.parquet`) | {val_tokens_written:,} | {val_pct_midtrain:.2f}% midtrain |

- Target: **{100 * RATIO_MIDTRAIN:.2f}% midtrain**; achieved **{ACHIEVED_PCT_PREFIX:.4f}%** over the
  first {int(PREFIX_TOKENS):,} tokens the run actually consumes
  ({ACHIEVED_PCT_PREFIX - 100 * RATIO_MIDTRAIN:+.4f} pp), **{ACHIEVED_PCT_WHOLE:.4f}%** over the whole upload.
- The ratio is enforced by a ratio-tracking interleave, so it holds over *every* prefix, not just
  the total.
- Supply: {TRAIN_TOKENS:,} train tokens vs {EXPECTED_CONSUMED_TOKENS:,} consumed
  ({TRAIN_TOKENS / EXPECTED_CONSUMED_TOKENS:.4f}x, {CACHE_MARGIN_FRAC:.0%} margin requested).

## Provenance

- Tokenizer: `{TOKENIZER_REPO}/{TOKENIZER_PREFIX}` (`{enc.name}`, vocab {VOCAB_SIZE:,}),
  fingerprint `{TOKENIZER_FINGERPRINT}`
- Midtrain source: `{MIDTRAIN_REPO}/{MIDTRAIN_SRC_PREFIX}` @ `{MIDTRAIN_REVISION}`
- Pretraining source: `{OLD_REPO}` @ `{OLD_REVISION}`, shards
  {OLD_SHARDS_USED[0]}..{OLD_SHARDS_USED[-1]} ({len(OLD_SHARDS_USED)} shards, starting
  {OLD_START_PCT:.0%} through the corpus). Disjoint from the base run's shards
  0..{BASE_RUN_OLD_SHARDS_UPTO} and from the other v2 mixture.
- Generator: `{GENERATOR_CODE_REPO}` @ `{GENERATOR_CODE_COMMIT}`
- Shuffle seed: `{RANDOM_SEED}` (windowed shuffle, {SHUFFLE_WINDOW_OLD_SHARDS}-shard window for the
  pretraining side, {SHUFFLE_WINDOW_MIDTRAIN_SHARDS} for midtrain)

## Repetition and separation

- Pretraining documents: {len(old_unique):,} unique, **no repeats** ({OLD_REPEATS} repeated ids,
  {OLD_HASH_DUPES} duplicate text hashes).
- Midtrain documents repeat by design: {len(mid_entries):,} emissions of
  {mid_unique_written:,} unique documents =
  **{MIDTRAIN_EPOCHS_TOKENS:.4f} effective epochs** by tokens ({MIDTRAIN_EPOCHS_DOCS:.4f} by
  documents), starting {MIDTRAIN_START_PCT:.0%} into the shuffled midtrain order and wrapping.
- Validation is a held-out shard ({val_docs_written} documents); its documents appear nowhere in
  training ({TRAIN_VAL_ID_OVERLAP} id overlaps, {TRAIN_VAL_HASH_OVERLAP} hash overlaps).
- Cross-source text overlap: {CROSS_SOURCE_HASH_OVERLAP}.
- Dedup at index time removed {dedup_old_internal:,} pretraining duplicates and
  {dedup_cross_source:,} documents that duplicated midtrain content.

## Layout

`{DATA_PREFIX}/shard_NNNNN.parquet` -- {len(plan)} train shards (indices 0..{len(plan) - 1}) plus the
validation shard at index {VAL_SHARD_IDX}. Per-shard sidecars with every document's id, source,
occurrence, chars, exact tokens and text hash live in `{SHARDMETA_PREFIX}/`.

| shard | split | docs | tokens | midtrain | sha256 |
|---|---|---|---|---|---|
{shard_table}

Updated: {updated_at}
'''

readme_local = os.path.join(meta_out, 'README.md')
with open(readme_local, 'w', encoding='utf-8') as f:
    f.write(readme)

final = robust_commit(
    [CommitOperationAdd(path_in_repo=f'{DST_PREFIX}/manifest.json', path_or_fileobj=manifest_local),
     CommitOperationAdd(path_in_repo=f'{DST_PREFIX}/README.md', path_or_fileobj=readme_local)],
    commit_message=f'v2/ratio_{RATIO_PCT} manifest + README '
                   f'({ACHIEVED_PCT_PREFIX:.3f}% midtrain over the consumed prefix)',
)
RUN_COMMITS.append(final.oid)
print(f"Uploaded manifest.json + README.md to {MIDTRAIN_REPO}/{DST_PREFIX}/")
print("\n" + "=" * 78)
print(f"IMMUTABLE COMMIT SHA (quote this, not `main`):\n  {final.oid}")
print(f"  dataset: {MIDTRAIN_REPO}/{DST_PREFIX}/data")
print(f"  revision URL: https://huggingface.co/datasets/{MIDTRAIN_REPO}/tree/{final.oid}/{DATA_PREFIX}")
print("=" * 78)

## 11. Read-back verification

Re-reads the uploaded folder as a consumer would: shard indices contiguous from 0, validation index
at or above the train-shard count, a `text` column, and the checksum of a sampled shard matching
the manifest.

In [ ]:
check = json.load(open(robust_hf_download(f'{DST_PREFIX}/manifest.json',
                                          repo_id=MIDTRAIN_REPO, token=HF_TOKEN)))
print(json.dumps({k: v for k, v in check.items() if k != 'shard_records'}, indent=2)[:4000])

uploaded = sorted((f for f in list_repo_paths(MIDTRAIN_REPO, token=HF_TOKEN)
                   if f.startswith(f'{DATA_PREFIX}/') and f.endswith('.parquet')),
                  key=shard_index_of)
indices = [shard_index_of(f) for f in uploaded]
print(f"\nUploaded: {len(uploaded)} shard(s), indices {indices[0]}..{indices[-1]}")
assert indices == list(range(len(indices))), "Shard indices are not contiguous from 0."
assert check['layout']['validation_shard_index'] >= check['layout']['n_train_shards'], (
    "Validation shard index must be >= the number of train shards.")
print(f"Contiguous 0..{indices[-1]} | {check['layout']['n_train_shards']} train + validation at "
      f"index {check['layout']['validation_shard_index']} -- OK")

sample_k = indices[len(indices) // 2]        # a mid-run shard, not just the first
sample_path = f'{DATA_PREFIX}/{shard_name(sample_k)}'
local = robust_hf_download(sample_path, repo_id=MIDTRAIN_REPO, token=HF_TOKEN)
tbl = pq.read_table(local)
assert tbl.schema.names == ['text'], f"Expected a single `text` column, got {tbl.schema.names}"
digest = sha256_file(local)
recorded = next(r['sha256'] for r in check['shard_records'] if r['shard_index'] == sample_k)
assert digest == recorded, f"sha256 mismatch for {shard_name(sample_k)}: {digest} != {recorded}"
lengths = [len(t) for t in tbl.column('text').to_pylist()]
print(f"{shard_name(sample_k)}: {len(lengths):,} docs, columns={tbl.schema.names}, sha256 matches "
      f"the manifest")
print(f"  text length: min={min(lengths):,} median={sorted(lengths)[len(lengths) // 2]:,} "
      f"max={max(lengths):,}")

print(f"\nratio_{RATIO_PCT} v2 complete: {ACHIEVED_PCT_PREFIX:.4f}% midtrain over the consumed "
      f"prefix, {TRAIN_TOKENS:,} train tokens, {len(plan)} train shards + validation.")
print(f"Hand back revision {RUN_COMMITS[-1]} (not `main`).")

## 12. Running the other mixture / notes

- **Second mixture:** in section 1 set `RATIO_MIDTRAIN = 0.60`,
  `EXPECTED_CONSUMED_TOKENS = 2_013_265_920`, `OLD_START_PCT = 0.90`, and run top to bottom again.
  Section 7 will refuse to proceed if its old-corpus shards collide with the r30 mixture's, so the
  two pretraining portions stay disjoint. Consider setting `MIDTRAIN_START_PCT` to a different
  offset if you want the two mixtures to enter the midtrain corpus at different points.
- **Reusing the calibration:** the measured chars/token printed by section 9 can be pinned in
  `CPT_OVERRIDE` for the next mixture, which makes its planned ratio land closer on the first pass
  and lets you drop `CALIB_SAMPLE_EVERY` to a larger number.
- **After a disconnect:** rerun top to bottom. Indexing re-reads from the HF cache, the plan is
  deterministic under the same config, and section 8 skips shards already uploaded after checking a
  sample of them against the plan. Sidecars let section 9 report exactly on shards it did not
  rebuild.
- **Changing any planning parameter** (seed, ratio, token targets, start percentages, shard size)
  changes what `shard_k` means. Wipe with section 1b before rebuilding -- section 8 will abort
  rather than mix two plans in one folder.
- **What the trainer sees:** `midtraining_anneal_ablation.ipynb` builds
  `DATASET_PREFIX = f'mixed/ratio_{RATIO_PCT}/data'`. Point it at `mixed/v2/ratio_<pct>/data` (and
  pin the revision from section 10) to consume these mixtures. It already treats the
  highest-numbered shard as validation, which is the held-out shard this notebook writes.